<!--
Copyright (c) 2026 OceanBase.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# 18 · 把 Skill 安装到 Agent 工作目录

服务端有 Skill，并不表示另一处 Agent 已能发现它。我们启动一个独立 Receiver 进程，让它拉取期望状态、下载并安装准确版本、回传结果，随后观察修改冲突和撤回。

无需模型。Receiver 使用真实产品实现，运行在本机的独立临时工作目录；这是两个进程之间的远程分发协议实验，不是跨机器网络验收。所有安装、冲突和删除均限定于本篇新建目录。第 22 篇会让真实 Agent 使用交付后的内容。

路线：制品 → target 注册与 enrollment → 发布 → Receiver 安装 → 本地修改冲突 → 撤回 → 凭证撤销。

In [ ]:
import sys
from pathlib import Path

from _tutorial import Tutorial, show

from powercontext.http import CreateScopeRequest

if not Path("_tutorial.py").is_file():
    sys.path.insert(0, str(Path.cwd() / "examples" / "jupyter"))
if previous_lab := globals().get("lab"):
    await previous_lab.close()
lab = await Tutorial.start("18", features=())
client = lab.client
assert client is not None
scope = await client.create_scope(
    CreateScopeRequest(
        title="订单 CSV 导入器 · 18", summary="本次教学实验的独立材料", idempotency_key=f"{lab.run_id}:main"
    )
)
scope_id = scope.scope_id

## 创建内容和一个接收目标

Target 表示一处 Agent 安装。一次性 enrollment code 用于换取 Receiver 凭证；下面只显示目标 ID，不显示凭证或 enrollment code。

In [ ]:
from powercontext.http import (
    ArtifactReference,
    CreateArtifactRequest,
    CreateRemoteSkillTargetRequest,
    EnrollRemoteSkillTargetRequest,
    ListRemoteSkillTargetsRequest,
    PublishRemoteSkillRequest,
)

created = await client.create_artifact(
    scope_id,
    CreateArtifactRequest.model_validate({
        "family": "skill",
        "content": {
            "name": "csv-delivery-check",
            "description": "Check amount precision for CSV changes.",
            "instructions": "Read the project constraints. Check ordinary and fractional-cent inputs. Report actual results.",
            "validation": ["Checks have actual results"],
        },
    }),
)
ref = ArtifactReference(family="skill", artifact_id=created.artifact_id, revision=created.revision)
enrollment = await client.create_remote_skill_target(
    CreateRemoteSkillTargetRequest(scope_id=scope_id, agent_kind="codex", display_name="教程独立 Receiver")
)
credential = await client.enroll_remote_skill_target(
    EnrollRemoteSkillTargetRequest(
        enrollment_code=enrollment.enrollment_code,
        installation_id=lab.run_id,
        receiver_version="0.1.0",
        workspace_name="tutorial-receiver",
    )
)
target_id = credential.target_id
receiver_workspace = lab.directory / "receiver-project"
receiver_workspace.mkdir()
publication = await client.publish_remote_skill(
    PublishRemoteSkillRequest(scope_id=scope_id, target_id=target_id, artifact=ref, expected_generation=None)
)
show({"目标": target_id, "目标目录": str(receiver_workspace), "发布已请求": True})

## 用独立进程完成一次同步

进程入口只负责配置传递，真正的对账、摘要校验、原子安装和回执都由 RemoteSkillReceiver 完成。凭证通过标准输入传入，不出现在命令行或配置文件中。

In [ ]:
import asyncio
import json

receiver_script = Path(_tutorial_file := sys.modules["_tutorial"].__file__).parent / "support" / "receiver_worker.py"


async def sync_receiver():
    process = await asyncio.create_subprocess_exec(
        sys.executable,
        str(receiver_script),
        stdin=asyncio.subprocess.PIPE,
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.PIPE,
    )
    config = {
        "server_url": lab.base_url,
        "target_id": target_id,
        "credential": credential.credential,
        "agent_kind": "codex",
        "workspace": str(receiver_workspace),
    }
    stdout, _stderr = await asyncio.wait_for(process.communicate(json.dumps(config).encode()), timeout=90)
    assert process.returncode == 0, "Receiver 进程失败，请查看本地服务状态"
    return json.loads(stdout)


result = await sync_receiver()
assert result["succeeded"] == 1 and result["failed"] == 0
installed = receiver_workspace / ".agents" / "skills" / "csv-delivery-check" / "SKILL.md"
assert installed.is_file()
original_bytes = installed.read_bytes()
print(installed.read_text())
status = await client.list_remote_skill_targets(ListRemoteSkillTargetsRequest(scope_id=scope_id, target_id=target_id))
show(status)

## 更新 Receiver 安装的版本

为标准技能包提出完整替换候选，检查 pending 时旧版本不变，批准后明确发布新版本，并通过 Receiver 核对实际文件。

标准包把校验说明保存在 SKILL.md 中。更新完整包时，应保留包内其他文件和元数据；不能只抽取旧字段、丢掉 package，再当成传统文本 Skill 写回。


In [ ]:
import base64
import io
import zipfile

from powercontext.http import ApproveArtifactCandidateRequest, GetSkillPackageRequest, ProposeSkillPackageRequest

package = await client.download_skill_package(GetSkillPackageRequest(scope_id=scope_id, artifact=ref))
updated_archive = io.BytesIO()
with (
    zipfile.ZipFile(io.BytesIO(base64.b64decode(package.archive_base64))) as before,
    zipfile.ZipFile(updated_archive, "w") as after,
):
    for item in before.infolist():
        content = before.read(item.filename)
        if item.filename == "SKILL.md":
            content += b"\n\n## Additional checks\nAlways check empty input and non-finite values.\n"
        after.writestr(item, content)
candidate = await client.propose_skill_package(
    ProposeSkillPackageRequest(
        scope_id=scope_id,
        archive_base64=base64.b64encode(updated_archive.getvalue()).decode(),
        target=ref,
        reason="保留完整包并增加空输入与非有限值检查步骤",
    )
)
assert candidate.status == "pending"
assert (await client.get_artifact(scope_id, "skill", ref.artifact_id)).revision == ref.revision
show({"候选状态": candidate.status, "待加入步骤": "Always check empty input and non-finite values."})
approved = await client.approve_artifact_candidate(
    ApproveArtifactCandidateRequest(
        scope_id=scope_id, candidate_id=candidate.candidate_id, expected_version=candidate.version
    )
)
new_ref = approved.result_artifact
assert new_ref and new_ref.artifact_id == ref.artifact_id and new_ref.revision > ref.revision
publication = await client.publish_remote_skill(
    PublishRemoteSkillRequest(
        scope_id=scope_id, target_id=target_id, artifact=new_ref, expected_generation=publication.generation
    )
)
update_result = await sync_receiver()
assert update_result["succeeded"] == 1 and update_result["failed"] == 0
assert installed.read_bytes() != original_bytes
assert "non-finite" in installed.read_text()
original_bytes = installed.read_bytes()
show({"receiver_revision": new_ref.revision, "updated_instructions_installed": True})

## Receiver 不应覆盖接收方自己的修改

接收方改动文件后，我们请求撤回。文件与已安装摘要不符时，Receiver 必须保留它并报告冲突。随后由本实验恢复自己修改的字节，再重试撤回。

In [ ]:
from powercontext.http import UnpublishRemoteSkillRequest

installed.write_bytes(original_bytes + b"\nLocal tutorial edit.\n")
withdrawal = await client.unpublish_remote_skill(
    UnpublishRemoteSkillRequest(
        scope_id=scope_id, target_id=target_id, artifact_id=ref.artifact_id, expected_generation=publication.generation
    )
)
conflict = await sync_receiver()
assert conflict["failed"] == 1 and installed.is_file()
assert b"Local tutorial edit" in installed.read_bytes()
print("撤回遇到本地修改：保留文件并报告冲突。")
installed.write_bytes(original_bytes)
removed = await sync_receiver()
assert removed["succeeded"] == 1 and not installed.exists()
print("恢复本次修改后重试：托管安装已撤回。")

## 撤销目标凭证

撤回包与撤销 Receiver 身份是不同操作。最后撤销 target，使这个实验凭证不再具备后续同步能力。

In [ ]:
from powercontext.http import RevokeRemoteSkillTargetRequest

latest = await client.list_remote_skill_targets(ListRemoteSkillTargetsRequest(scope_id=scope_id, target_id=target_id))
revoked = await client.revoke_remote_skill_target(
    RevokeRemoteSkillTargetRequest(
        scope_id=scope_id, target_id=target_id, expected_generation=latest.targets[0].target.generation
    )
)
assert revoked.state == "revoked"
show({"目标状态": revoked.state, "安装已删除": not installed.exists()})

## 练习与验收

第一次成功安装后再同步一次，应不重复安装。接着尝试发布同一 Skill 的新版本，检查实际文件变化和回执；第 22 篇会把交付接到下一次 Agent 任务。

接下来阅读 [19_mcp_agent_tools.ipynb](19_mcp_agent_tools.ipynb)。

最后关闭服务。实验文件保留在本次 `.powercontext/` 目录，便于复查。

In [ ]:
await lab.close()
print("本篇 Server 已关闭。")